# Using keras tuner to find optimal hyperparameters for the multi layer perceptron

In [1]:
import os
import sys
from glob import glob

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import keras
import keras_tuner as kt
import tensorflow as tf
import tensorboard as tb
import numpy as np
import pandas as pd

from src.loaders import load_data_array
from utils import create_submission, get_run_logdir

2025-08-02 13:53:45.200125: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-02 13:53:45.345598: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754164425.404810   35487 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754164425.428344   35487 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754164425.560629   35487 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [7]:
def data_generator(sample_ids, features_subset):
    """Generator that yields one sample at a time"""
    wide_data = features_subset.values  # Your stats data
    
    for i, sample_id in enumerate(sample_ids):
        # Load one sample at a time
        X_deep_single, y_single = load_data_array([sample_id])
        
        # Convert to float32
        X_deep_single = X_deep_single[0].astype(np.float32)  # Remove batch dimension
        y_single = y_single[0].astype(np.float32)
        X_wide_single = wide_data[i].astype(np.float32)
        
        yield (X_wide_single, X_deep_single), y_single

features = pd.read_csv('../data/processed/final_features.csv')
is_train = features['split'] == 'train'

# Create datasets
train_ids = features.loc[is_train, 'sample_id'].reset_index(drop=True)
val_ids = features.loc[~is_train, 'sample_id'].reset_index(drop=True)

X_wide_train = features.loc[is_train].drop(columns=['sample_id', 'split', 'env_max', 'dom_freq'])
X_wide_val = features.loc[~is_train].drop(columns=['sample_id', 'split', 'env_max', 'dom_freq'])

# Create TensorFlow datasets
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(train_ids, X_wide_train),
    output_signature=(
        (tf.TensorSpec(shape=(6,), dtype=tf.float32),           # wide input
         tf.TensorSpec(shape=(5, 10001, 31), dtype=tf.float32)), # deep input  
        tf.TensorSpec(shape=(300, 1259), dtype=tf.float32)      # target
    )
)

val_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(val_ids, X_wide_val),
    output_signature=(
        (tf.TensorSpec(shape=(6,), dtype=tf.float32),
         tf.TensorSpec(shape=(5, 10001, 31), dtype=tf.float32)),
        tf.TensorSpec(shape=(300, 1259), dtype=tf.float32)
    )
)

# Optimize the pipeline
BATCH_SIZE = 8  # Start small due to large input size
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✅ TensorFlow datasets created!")

✅ TensorFlow datasets created!


In [8]:
def build_model(hp):
    n_hidden = hp.Int('n_hidden', min_value = 0, max_value = 4, default = 3)
    n_neurons = hp.Int('n_neurons', min_value = 30, max_value = 128)
    learning_rate = hp.Float('learning_rate', min_value = 1e-2, max_value = 1e-1, sampling = 'log')

    # Inputs
    input_wide = keras.layers.Input(shape = [6], name='stats')
    input_deep = keras.layers.Input(shape = [5, 10001, 31], name = 'gathers')

    input_deep_flatten = keras.layers.Flatten()(input_deep)

    x = input_deep_flatten
    # Add n_hidden layers, each with n_neurons
    for i in range(n_hidden):
        x = keras.layers.Dense(n_neurons, activation='relu')(x)

    # Concatenation
    concatenation_layer = keras.layers.Concatenate()([input_wide, x])

    # Final dense layer
    hidden4 = keras.layers.Dense(377700)(concatenation_layer)

    # Output reshape
    output = keras.layers.Reshape([300, 1259])(hidden4)

    # Create the model
    model = keras.Model(inputs = [input_wide, input_deep], outputs = output)

    # Optimizer selection
    optimizer = hp.Choice('optimizer', values = ['sgd', 'adam', 'adamw', 'rmsprop'])
    if optimizer == 'sgd':
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer == 'adamw':
        optimizer = keras.optimizers.AdamW(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics = ['mape', 'mae']
    )

    return model

In [10]:
# Remember we need the following callbacks: tensoboard, reduce lr on plateau and model checkpoints
# Nevermind checkpoint will be used only when we get the best hyperparameters
# Also tensorboard logs will be used with best hyperparameters!

reduceLR_callback = keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5)

In [11]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=30,
    directory='../tuning',
    project_name='velocity_mlp_v2'
)

Reloading Tuner from ../tuning/velocity_mlp_v2/tuner0.json


In [12]:
tuner.search(
    train_dataset,
    validation_data = val_dataset,
    epochs = 5,
    callbacks = [reduceLR_callback]
)


Search: Running Trial #5

Value             |Best Value So Far |Hyperparameter
1                 |4                 |n_hidden
242               |76                |n_neurons
0.006214          |0.00017676        |learning_rate
adam              |rmsprop           |optimizer



2025-08-02 15:33:12.105893: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.40GiB (rounded to 1500550144)requested by op StatelessRandomUniformV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-08-02 15:33:12.106029: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1058] BFCAllocator dump for GPU_0_bfc
2025-08-02 15:33:12.106075: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (256): 	Total Chunks: 3074, Chunks in use: 3074. 768.5KiB allocated for chunks. 768.5KiB in use in bin. 12.0KiB client-requested in use in bin.
2025-08-02 15:33:12.106079: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (512): 	Total Chunks: 10, Chunks in use: 10. 5.5KiB allocated for chunks. 5.5KiB in use in bin. 3.5KiB client-requested 

RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "/home/berns/miniconda3/envs/seismic_env/lib/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/home/berns/miniconda3/envs/seismic_env/lib/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/berns/miniconda3/envs/seismic_env/lib/python3.11/site-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copied_kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/berns/miniconda3/envs/seismic_env/lib/python3.11/site-packages/keras_tuner/src/engine/tuner.py", line 232, in 

InUse at 658416f00 of size 256 next 2761
2025-08-02 15:33:12.109450: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 658417000 of size 256 next 2760
2025-08-02 15:33:12.109451: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 658417100 of size 256 next 2765
2025-08-02 15:33:12.109452: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 658417200 of size 256 next 2764
2025-08-02 15:33:12.109453: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 658417300 of size 256 next 2763
2025-08-02 15:33:12.109454: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 658417400 of size 256 next 2768
2025-08-02 15:33:12.109455: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 658417500 of size 256 next 2767
2025-08-02 15:33:12.109456: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 658417600 of size 256 next 2766
2025-08-02 15:33:12.109457: I external/loc